# EfficientNet-B0 촬영 각도 분류기: 로컬 학습

이 노트북은 확정된 `final_angle` 라벨을 검증하고 ImageFolder 데이터셋 구성, 학습, 평가를 순서대로 실행합니다.

In [ ]:
from pathlib import Path
import sys, torch, torchvision, PIL

PROJECT_ROOT = Path(r'C:\dev\final_1_team\apps\api\food-image-cleanup-pipeline')
assert PROJECT_ROOT.is_dir(), f'프로젝트 경로를 확인하세요: {PROJECT_ROOT}'
print('Python:', sys.version)
print('Torch:', torch.__version__, 'Torchvision:', torchvision.__version__)
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available(): print('GPU:', torch.cuda.get_device_name(0))
print('Pillow:', PIL.__version__)

In [ ]:
# 빈 라벨, 오타, 누락 클래스가 있으면 이 단계가 중단되며 CSV를 먼저 수정해야 합니다.
import os, subprocess
subprocess.run([sys.executable, '-m', 'scripts.prepare_angle_classification_dataset'], cwd=PROJECT_ROOT, check=True)

In [ ]:
# -u와 Popen을 사용해 학습 중 배치 진행률·검증 지표를 즉시 표시합니다.
command = [sys.executable, '-u', '-m', 'scripts.train_efficientnet_b0_angle_classifier', '--epochs', '50', '--batch-size', '32']
environment = os.environ.copy()
environment['PYTHONUNBUFFERED'] = '1'
process = subprocess.Popen(command, cwd=PROJECT_ROOT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=environment)
assert process.stdout is not None
for line in process.stdout:
    print(line, end='', flush=True)
if process.wait() != 0:
    raise RuntimeError('EfficientNet-B0 학습이 실패했습니다.')

In [ ]:
weights = PROJECT_ROOT / 'runs/efficientnet_b0_angle/best.pt'
subprocess.run([sys.executable, '-m', 'scripts.evaluate_efficientnet_b0_angle_classifier', '--weights', str(weights)], cwd=PROJECT_ROOT, check=True)